# 6. Export generated annotations in MEVA-style JSON

Convert the model-generated event annotations and their YOLO spatial coordinates into four JSON files per video under `data/processed/<video-name>/meva_export/`:

- `<video-name>.activities.json` — generated events and their participating object IDs.
- `<video-name>.geom.json` — frame-level bounding boxes for objects involved in events.
- `<video-name>.types.json` — object IDs, detected labels, coarse MEVA-style types, confidence, and observed time range.
- `<video-name>.metadata.json` — source path, duration, frame rate, resolution, camera ID, scene, and capture-time range parsed from the video name.

These are **model-generated annotations**, not MEVA ground truth: event labels retain the local model's vocabulary and every activity is marked `src_status: generated`.

In [7]:
import hashlib
import json
import re
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import cv2

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "notebooks").is_dir() and (parent / "annotation_pipeline").is_dir()
)
RAW_VIDEOS = PROJECT_ROOT / "data" / "raw" / "videos"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

def parse_video_name(video_name):
    """Extract the capture interval, scene, and camera ID embedded in MEVA names."""
    match = re.fullmatch(
        r"(?P<date>\d{4}-\d{2}-\d{2})\.(?P<start>\d{2}-\d{2}-\d{2})\."
        r"(?P<end>\d{2}-\d{2}-\d{2})\.(?P<scene>[^.]+)\.(?P<camera>[^.]+)",
        video_name,
    )
    if not match:
        return {"video_name": video_name}

    values = match.groupdict()
    capture_start = datetime.strptime(
        f"{values['date']} {values['start']}", "%Y-%m-%d %H-%M-%S"
    )
    capture_end = datetime.strptime(
        f"{values['date']} {values['end']}", "%Y-%m-%d %H-%M-%S"
    )
    return {
        "video_name": video_name,
        "scene": values["scene"],
        "camera_id": values["camera"],
        "capture_start_local": capture_start.isoformat(),
        "capture_end_local": capture_end.isoformat(),
        "capture_time_zone": "unknown",
    }


def find_source_video(video_name):
    """Support both original AVI names and VideoMeta's optional .r13 suffix."""
    candidates = [RAW_VIDEOS / f"{video_name}.avi", RAW_VIDEOS / f"{video_name}.r13.avi"]
    return next((path for path in candidates if path.is_file()), None)


event_paths = sorted(PROCESSED_DIR.glob("*/window_event_annotations.json"))
if not event_paths:
    raise FileNotFoundError(f"No event annotations found in {PROCESSED_DIR}")

for event_path in event_paths:
    video_name = event_path.parent.name
    spatial_path = event_path.with_name("yolo_spatial_coordinates.json")
    if not spatial_path.is_file():
        print(f"Skipping {video_name}: missing {spatial_path.name}")
        continue

    window_annotations = json.loads(event_path.read_text(encoding="utf-8"))
    spatial_windows = json.loads(spatial_path.read_text(encoding="utf-8"))
    export_dir = event_path.parent / "meva_export"
    export_dir.mkdir(exist_ok=True)

    activities = []
    event_object_ids = set()
    object_descriptions = defaultdict(lambda: defaultdict(set))
    activity_id = 0
    for window in window_annotations:
        start_frame = min(
            (detection["frame_index"] for item in spatial_windows
             if item["start_seconds"] == window["start_seconds"]
             for detection in item["detections"]),
            default=None,
        )
        end_frame = max(
            (detection["frame_index"] for item in spatial_windows
             if item["start_seconds"] == window["start_seconds"]
             for detection in item["detections"]),
            default=start_frame,
        )
        if start_frame is None:
            continue
        for event in window.get("events", []):
            for object_item in event.get("involved_objects", []):
                if object_item.get("id") is not None and object_item.get("physical_details"):
                    object_descriptions[str(object_item["id"])][object_item.get("label", "unknown")].add(
                        object_item["physical_details"]
                    )
            actor_ids = sorted(
                {str(item["id"]) for item in event.get("involved_objects", []) if item.get("id") is not None}
            )
            event_object_ids.update(actor_ids)
            activities.append(
                {
                    "event_id": hashlib.sha256(
                        f"{video_name}:{activity_id}".encode("utf-8")
                    ).hexdigest(),
                    "event_name": event.get("event_name"),
                    "description": event.get("description"),
                    "actors": [{"object_id": object_id} for object_id in actor_ids],
                    "start_frame": start_frame,
                    "end_frame": end_frame,
                    "start_seconds": window["start_seconds"],
                    "end_seconds": window["end_seconds"],
                    "source_status": "generated",
                }
            )
            activity_id += 1

    detections_by_object = defaultdict(list)
    seen_geometry = set()
    geometries = []
    geometry_id = 0
    for window in spatial_windows:
        for detection in window.get("detections", []):
            object_id = str(detection["object_id"])
            if object_id not in event_object_ids:
                continue
            box = [round(float(value)) for value in detection["box_xyxy"]]
            geometry_key = (object_id, detection["frame_index"], tuple(box))
            if geometry_key in seen_geometry:
                continue
            seen_geometry.add(geometry_key)
            detections_by_object[object_id].append(detection)
            geometries.append(
                {
                    "geometry_id": geometry_id,
                    "object_id": object_id,
                    "frame_index": int(detection["frame_index"]),
                    "timestamp_seconds": detection["timestamp_seconds"],
                    "label": detection["label"],
                    "confidence": detection["confidence"],
                    "spatial_position": detection["spatial_position"],
                    "bounding_box_pixels": {
                        "x_min": box[0],
                        "y_min": box[1],
                        "x_max": box[2],
                        "y_max": box[3],
                    },
                }
            )
            geometry_id += 1

    types = []
    for object_id, detections in sorted(detections_by_object.items(), key=lambda item: int(item[0])):
        timestamps = [detection["timestamp_seconds"] for detection in detections]
        detections_by_label = defaultdict(list)
        for detection in detections:
            detections_by_label[detection["label"]].append(detection)
        types.append(
            {
                "object_id": object_id,
                "detection_count": len(detections),
                "first_seen_seconds": min(timestamps),
                "last_seen_seconds": max(timestamps),
                "labels": [
                    {
                        "label": label,
                        "descriptions": sorted(object_descriptions[object_id][label]),
                        "detection_count": len(label_detections),
                        "confidence": {
                            "minimum": round(
                                min(item["confidence"] for item in label_detections), 4
                            ),
                            "average": round(
                                sum(item["confidence"] for item in label_detections)
                                / len(label_detections),
                                4,
                            ),
                            "maximum": round(
                                max(item["confidence"] for item in label_detections), 4
                            ),
                        },
                        "spatial_positions": sorted(
                            {item["spatial_position"] for item in label_detections}
                        ),
                    }
                    for label, label_detections in sorted(detections_by_label.items())
                ],
            }
        )

    metadata = parse_video_name(video_name)
    source_video = find_source_video(video_name)
    metadata.update(
        {
            "video_id": hashlib.sha256(f"video:{video_name}".encode("utf-8")).hexdigest(),
            "annotation_format": "MEVA-style generated export",
            "event_annotation_path": str(event_path.relative_to(PROJECT_ROOT)),
            "spatial_annotation_path": str(spatial_path.relative_to(PROJECT_ROOT)),
            "source_video_path": str(source_video.relative_to(PROJECT_ROOT)) if source_video else None,
            "source_video_available": source_video is not None,
            "activity_count": len(activities),
            "object_count": len(types),
            "geometry_count": len(geometries),
        }
    )
    if source_video:
        capture = cv2.VideoCapture(str(source_video))
        metadata["duration_seconds"] = round(
            capture.get(cv2.CAP_PROP_FRAME_COUNT) / capture.get(cv2.CAP_PROP_FPS), 3
        )
        metadata["frame_count"] = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
        metadata["fps"] = round(capture.get(cv2.CAP_PROP_FPS), 6)
        metadata["frame_width"] = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
        metadata["frame_height"] = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
        metadata["fourcc"] = "".join(
            chr((int(capture.get(cv2.CAP_PROP_FOURCC)) >> 8 * index) & 0xFF)
            for index in range(4)
        )
        capture.release()

    for suffix, payload in {
        "activities": activities,
        "geom": geometries,
        "types": types,
        "metadata": metadata,
    }.items():
        output_path = export_dir / f"{video_name}.{suffix}.json"
        output_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

    print(
        f"Exported {video_name}: {len(activities)} activities, "
        f"{len(types)} objects, and {len(geometries)} geometry records"
    )


Exported 2018-03-05.13-15-00.13-20-00.bus.G340: 20 activities, 20 objects, and 3483 geometry records
Exported 2018-03-05.13-15-00.13-20-00.bus.G505: 17 activities, 16 objects, and 3893 geometry records
Exported 2018-03-05.13-15-00.13-20-00.bus.G506: 4 activities, 4 objects, and 723 geometry records
Exported 2018-03-05.13-15-00.13-20-00.hospital.G341: 13 activities, 10 objects, and 2114 geometry records
Exported 2018-03-05.13-15-00.13-20-00.school.G423: 0 activities, 0 objects, and 0 geometry records
Exported 2018-03-05.13-15-00.13-20-00.school.G424: 32 activities, 38 objects, and 6233 geometry records
Exported 2018-03-05.13-15-01.13-20-01.school.G328: 9 activities, 35 objects, and 6250 geometry records
Exported 2018-03-05.13-20-00.13-25-00.bus.G340: 10 activities, 8 objects, and 1009 geometry records
Exported 2018-03-05.13-20-00.13-25-00.school.G336: 13 activities, 15 objects, and 3783 geometry records
Exported 2018-03-05.13-20-00.13-25-00.school.G423: 5 activities, 5 objects, and 149 